# Satellite close-approach hackathon

## Aim of this exercise

You are given a **catalog of satellites** (orbital data at known times). Your job is to write an algorithm that finds **close approaches**: pairs of objects that come near each other while you **propagate** their orbits forward in time.

### What you are given

| Item | What it is |
|------|------------|
| `spacetrack_data.json` | Catalog with ~28k objects whose orbit data (epoch) is from **31 May 2026** or **1 June 2026** |
| `conjunction_toolkit/` | Library to load the catalog, propagate positions, run a naive baseline, plot, and verify claims |
| `student_solution.py` | Empty function you must implement |
| This notebook | Walkthrough + the same verifier for baseline and your code |

Each catalog row includes an object id, name, epoch (when the orbit was measured), and orbital elements used by SGP4 to predict future positions.

### End goal

1. Implement `find_close_approaches` in `student_solution.py`.
2. For a chosen time range (`propagate_from_utc` → `propagate_until_utc`), return every pair that comes within a distance threshold.
3. Each answer is a **claim**: two object ids + time of closest approach + miss distance (km).
4. Pass the shared **verifier**, which re-propagates both objects and checks your time and distance (default: within ~100 m and ~5 seconds).

Beat the **naive baseline** (check every pair on a coarse time grid) on speed and/or quality of verified claims.

**Files you edit / run**

| File | Role |
|------|------|
| `conjunction_tutorial.ipynb` | Run top → bottom |
| `student_solution.py` | **Your algorithm** |
| `spacetrack_data.json` | Input catalog |

Do not edit `conjunction_toolkit/` unless an organizer asks you to.

## 0. Setup

In [ ]:
import sys
import subprocess
from pathlib import Path

pkgs = [
    "skyfield>=1.48",
    "sgp4>=2.23",
    "numpy>=1.26",
    "scipy>=1.11",
    "plotly>=5.18",
    "pandas>=2.1",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

ROOT = Path.cwd()
if not (ROOT / "conjunction_toolkit").exists():
    raise FileNotFoundError("Open / run this notebook from the project root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Ready. Project root:", ROOT)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Callable, Dict, List, Optional, Sequence
import importlib
import time

from skyfield.api import EarthSatellite

from conjunction_toolkit import (
    ConjunctionClaim,
    VerifyConfig,
    catalog_to_satellites,
    closest_approach_on_grid,
    improve_closest_approach_estimate,
    load_default_catalog,
    plot_pair_with_distance,
    plot_trajectories,
    propagate_positions,
    save_html,
    screen_pairs,
    time_grid,
    verify_claim,
)
from conjunction_toolkit.propagate import datetimes_of

OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Toolkit cheat sheet (what you may call)

| Function | What it does |
|----------|----------------|
| `load_default_catalog()` | Load `spacetrack_data.json` |
| `catalog_to_satellites(catalog, norad_ids=...)` | Build objects you can propagate |
| `time_grid(from, until, step_seconds)` | Sample times from start **until** end |
| `propagate_positions(sat, times)` | Positions (km) at those times |
| `closest_approach_on_grid(pos_a, pos_b, times)` | Among **those samples only**, when were they nearest? |
| `improve_closest_approach_estimate(sat_a, sat_b, rough_time)` | **Zoom in** around a rough closest time with a finer search |
| `screen_pairs(...)` | Naive baseline: every pair × every sample |
| `verify_claim(claim, satellites)` | Independent check of one claim |
| `plot_trajectories` / `plot_pair_with_distance` / `save_html` | Plots |

**Claim fields:** `norad_a`, `norad_b`, `tca_utc` (= time of closest approach in UTC), `min_distance_km`.

## 2. Load the catalog

In [ ]:
catalog = load_default_catalog()

print(f"Source  : {catalog.source_path}")
print(f"Objects : {len(catalog)}")

epochs = [obj.epoch_utc for obj in catalog]
print(f"Epochs  : {min(epochs).date()} → {max(epochs).date()}  (data dates, not your propagate range)")

print("\nSample objects:")
for object_id in catalog.ids()[:5]:
    obj = catalog[object_id]
    print(f"  id={obj.norad_cat_id:>6}  {obj.object_name:<28}  epoch={obj.epoch_utc.date()}")

## 3. Propagate and plot a few orbits

In [ ]:
demo_ids: list[int] = []
for name in ("ISS (ZARYA)", "HST", "CSS (TIANHE)"):
    matches = list(catalog.filter_by_name(name))
    if matches:
        matches.sort(key=lambda obj: obj.epoch_utc, reverse=True)
        demo_ids.append(matches[0].norad_cat_id)

if len(demo_ids) < 2:
    recent = sorted(catalog, key=lambda obj: obj.epoch_utc, reverse=True)
    demo_ids = [obj.norad_cat_id for obj in recent[:3]]

demo_ids = demo_ids[:3]
demo_satellites = catalog_to_satellites(catalog, norad_ids=demo_ids)

propagate_from_utc = max(catalog[i].epoch_utc for i in demo_ids)
PROPAGATE_FOR_HOURS = 6
propagate_until_utc = propagate_from_utc + timedelta(hours=PROPAGATE_FOR_HOURS)
orbit_times = time_grid(propagate_from_utc, propagate_until_utc, step_seconds=60)

print("Objects:", [(i, demo_satellites[i].name) for i in demo_ids])
print(f"Propagate from : {propagate_from_utc.isoformat()}")
print(f"Propagate until: {propagate_until_utc.isoformat()}  ({PROPAGATE_FOR_HOURS} hours ahead)")

fig = plot_trajectories(demo_satellites, orbit_times, title="Sample orbits")
save_html(fig, OUTPUT_DIR / "notebook_trajectories.html")
fig.show()

## 4. Closest approach for one pair (coarse sample → improved estimate)

1. **`closest_approach_on_grid`** — among the discrete samples only (e.g. every 60 s in the plot), pick the sample where distance was smallest. That time is a **rough** closest time.
2. **`improve_closest_approach_estimate`** — starting from that rough time, re-propagate on a much finer grid (seconds) to get a better closest time and distance. Verification needs this accuracy.

In [ ]:
id_a, id_b = demo_ids[0], demo_ids[1]
sat_a, sat_b = demo_satellites[id_a], demo_satellites[id_b]

positions_a = propagate_positions(sat_a, orbit_times)
positions_b = propagate_positions(sat_b, orbit_times)
sample_times = datetimes_of(orbit_times)

# Step 1: best among the coarse samples only
rough_closest_time, rough_distance_km, _ = closest_approach_on_grid(
    positions_a, positions_b, sample_times
)

# Step 2: zoom in around that rough time
improved_closest_time, improved_distance_km = improve_closest_approach_estimate(
    sat_a, sat_b, rough_closest_time
)

print(f"Pair {id_a} – {id_b}")
print(f"  Rough (best sample on the time grid): {rough_distance_km:.3f} km at {rough_closest_time.isoformat()}")
print(f"  Improved (finer search nearby)      : {improved_distance_km:.3f} km at {improved_closest_time.isoformat()}")

## 5. Naive baseline (check every pair)

1. Choose how long to propagate: **`PROPAGATE_FOR_HOURS`** (here 6 hours).
2. Choose the sample jump: **`TIME_STEP_MINUTES`** (here 30 minutes).
3. For every unique pair, measure distance at each sample from `propagate_from_utc` **until** `propagate_until_utc`.
4. Keep pairs whose closest sample distance ≤ threshold.

| Setting | Example | Meaning |
|---------|---------|---------|
| `PROPAGATE_FOR_HOURS` | **6** | Propagate / search for 6 hours ahead |
| `TIME_STEP_MINUTES` | **30** | Check positions every 30 minutes |
| Threshold | **100 km** | Keep pairs within 100 km on those samples |
| Objects | **40** | $N$ objects ⇒ $N(N-1)/2$ pairs |

With 6 hours and a 30-minute step you get about **13** samples per object.

In [ ]:
PROPAGATE_FOR_HOURS = 6
TIME_STEP_MINUTES = 30
TIME_STEP_SECONDS = TIME_STEP_MINUTES * 60
CLOSE_APPROACH_THRESHOLD_KM = 100.0
N_OBJECTS = 40

pool = [obj for obj in catalog if "STARLINK" in obj.object_name.upper()]
pool.sort(key=lambda obj: obj.norad_cat_id)
subset = pool[:N_OBJECTS]
subset_ids = [obj.norad_cat_id for obj in subset]
satellites = catalog_to_satellites(catalog, norad_ids=subset_ids)

epochs = sorted(obj.epoch_utc for obj in subset)
propagate_from_utc = epochs[len(epochs) // 2]
propagate_until_utc = propagate_from_utc + timedelta(hours=PROPAGATE_FOR_HOURS)

n_pairs = len(subset_ids) * (len(subset_ids) - 1) // 2
n_samples = int(PROPAGATE_FOR_HOURS * 3600 / TIME_STEP_SECONDS) + 1

print("Baseline setup")
print(f"  Objects                 : {len(subset_ids)}")
print(f"  Pairs to check          : {n_pairs}")
print(f"  Propagate for           : {PROPAGATE_FOR_HOURS} hours")
print(f"  Propagate from (UTC)    : {propagate_from_utc.isoformat()}")
print(f"  Propagate until (UTC)   : {propagate_until_utc.isoformat()}")
print(f"  Time step               : {TIME_STEP_MINUTES} minutes (= {TIME_STEP_SECONDS} s)")
print(f"  Samples along that span : ~{n_samples}")
print(f"  Distance threshold      : {CLOSE_APPROACH_THRESHOLD_KM} km")

In [ ]:
def naive_baseline_find_close_approaches(
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    *,
    time_step_seconds: float,
    close_approach_threshold_km: float,
):
    """Same interface as student_solution.find_close_approaches."""
    return screen_pairs(
        satellites,
        propagate_from_utc,
        propagate_until_utc,
        step_seconds=time_step_seconds,
        threshold_km=close_approach_threshold_km,
        algorithm_id="naive_baseline",
    )


print("Running naive baseline…")
t0 = time.perf_counter()
baseline_raw = naive_baseline_find_close_approaches(
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    time_step_seconds=TIME_STEP_SECONDS,
    close_approach_threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
)
print(f"Done in {time.perf_counter() - t0:.2f} s")
print(f"Close approaches on the {TIME_STEP_MINUTES}-minute samples: {len(baseline_raw)}")
for claim in baseline_raw[:5]:
    print(
        f"  {claim.norad_a}–{claim.norad_b}: "
        f"{claim.min_distance_km:.3f} km at {claim.tca_utc.isoformat()}"
    )

## 6. Your solution — edit `student_solution.py`

```python
def find_close_approaches(
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    *,
    time_step_seconds: float,
    close_approach_threshold_km: float,
) -> list[ConjunctionClaim]:
    ...
```

It currently raises `NotImplementedError` on purpose.

In [ ]:
import student_solution

importlib.reload(student_solution)
print("Loaded:", Path(student_solution.__file__).resolve())

try:
    student_solution.find_close_approaches(
        satellites,
        propagate_from_utc,
        propagate_until_utc,
        time_step_seconds=TIME_STEP_SECONDS,
        close_approach_threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
    )
except NotImplementedError as exc:
    print("Stub OK (not implemented yet):", exc)

## 7. Verifier (same report for baseline and your code)

Prints **detected / verified OK / failed / runtime**.  
Before checking, claims are improved with `improve_closest_approach_estimate` so coarse sample times get a fair chance.

In [ ]:
@dataclass
class EvaluationReport:
    algorithm_name: str
    close_approaches_detected: int
    claims_verified_ok: int
    claims_failed: int
    runtime_seconds: float
    messages: List[str] = field(default_factory=list)
    claims: List[ConjunctionClaim] = field(default_factory=list)

    def print_summary(self) -> None:
        print(f"=== {self.algorithm_name} ===")
        print(f"  Close approaches detected : {self.close_approaches_detected}")
        print(f"  Verified OK               : {self.claims_verified_ok}")
        print(f"  Failed verification       : {self.claims_failed}")
        print(f"  Runtime                   : {self.runtime_seconds:.2f} s")
        for msg in self.messages:
            print(f"  - {msg}")


def improve_claims(
    claims: Sequence[ConjunctionClaim],
    satellites: Dict[int, EarthSatellite],
    *,
    algorithm_id: Optional[str] = None,
) -> List[ConjunctionClaim]:
    """Zoom in on each claim's rough closest time before verification."""
    improved: List[ConjunctionClaim] = []
    for claim in claims:
        closest_time, closest_distance_km = improve_closest_approach_estimate(
            satellites[claim.norad_a],
            satellites[claim.norad_b],
            claim.tca_utc,
        )
        improved.append(
            ConjunctionClaim(
                norad_a=claim.norad_a,
                norad_b=claim.norad_b,
                tca_utc=closest_time,
                min_distance_km=closest_distance_km,
                algorithm_id=algorithm_id or claim.algorithm_id,
            )
        )
    improved.sort(key=lambda c: c.min_distance_km)
    return improved


def evaluate_finder(
    find_close_approaches: Callable[..., List[ConjunctionClaim]],
    satellites: Dict[int, EarthSatellite],
    propagate_from_utc: datetime,
    propagate_until_utc: datetime,
    *,
    time_step_seconds: float,
    close_approach_threshold_km: float,
    algorithm_name: str,
    improve_before_verify: bool = True,
) -> EvaluationReport:
    config = VerifyConfig(max_miss_distance_km=close_approach_threshold_km)

    t0 = time.perf_counter()
    raw_claims = find_close_approaches(
        satellites,
        propagate_from_utc,
        propagate_until_utc,
        time_step_seconds=time_step_seconds,
        close_approach_threshold_km=close_approach_threshold_km,
    )
    runtime_seconds = time.perf_counter() - t0

    claims = list(raw_claims)
    if improve_before_verify and claims:
        claims = improve_claims(claims, satellites, algorithm_id=algorithm_name)

    ok_count = 0
    fail_count = 0
    messages: List[str] = []
    for claim in claims:
        result = verify_claim(claim, satellites, config=config)
        if result.ok:
            ok_count += 1
        else:
            fail_count += 1
            reason = result.messages[0] if result.messages else "verification failed"
            messages.append(
                f"FAIL {claim.norad_a}–{claim.norad_b}: "
                f"{claim.min_distance_km:.3f} km @ {claim.tca_utc.isoformat()} ({reason})"
            )

    return EvaluationReport(
        algorithm_name=algorithm_name,
        close_approaches_detected=len(raw_claims),
        claims_verified_ok=ok_count,
        claims_failed=fail_count,
        runtime_seconds=runtime_seconds,
        messages=messages,
        claims=claims,
    )


print("Evaluator ready.")

In [ ]:
baseline_report = evaluate_finder(
    naive_baseline_find_close_approaches,
    satellites,
    propagate_from_utc,
    propagate_until_utc,
    time_step_seconds=TIME_STEP_SECONDS,
    close_approach_threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
    algorithm_name="naive_baseline",
)
baseline_report.print_summary()

In [ ]:
importlib.reload(student_solution)

try:
    student_report = evaluate_finder(
        student_solution.find_close_approaches,
        satellites,
        propagate_from_utc,
        propagate_until_utc,
        time_step_seconds=TIME_STEP_SECONDS,
        close_approach_threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
        algorithm_name="student_solution",
    )
    student_report.print_summary()
except NotImplementedError as exc:
    print("Implement student_solution.py first.")
    print(exc)
    student_report = None

In [ ]:
if student_report is not None:
    print(f"{'algorithm':<20} {'detected':>10} {'verified_ok':>12} {'failed':>8} {'seconds':>10}")
    for report in (baseline_report, student_report):
        print(
            f"{report.algorithm_name:<20} "
            f"{report.close_approaches_detected:>10} "
            f"{report.claims_verified_ok:>12} "
            f"{report.claims_failed:>8} "
            f"{report.runtime_seconds:>10.2f}"
        )
else:
    print("Fill in student_solution.py, then re-run the student verifier cell.")

### Optional: plot one close approach

In [ ]:
report_to_plot = baseline_report  # switch to student_report when ready

if report_to_plot.claims:
    best = report_to_plot.claims[0]
    zoom = time_grid(
        best.tca_utc - timedelta(minutes=45),
        best.tca_utc + timedelta(minutes=45),
        step_seconds=30.0,
    )
    fig = plot_pair_with_distance(
        satellites[best.norad_a],
        satellites[best.norad_b],
        zoom,
        tca=best.tca_utc,
        norad_a=best.norad_a,
        norad_b=best.norad_b,
        threshold_km=CLOSE_APPROACH_THRESHOLD_KM,
        title=f"Close approach {best.norad_a}–{best.norad_b}",
    )
    save_html(fig, OUTPUT_DIR / "notebook_pair.html")
    fig.show()
else:
    print("No claims to plot.")